# VoiceMOS 2026 — Client gọi **API service 3 track** từ Kaggle

Notebook này **chỉ là client**: gửi audio tới API service (FastAPI) đã deploy trên Hugging Face Space, nhận điểm về. **Không nạp model, không cần GPU.**

> Service: `https://tranminhtoan140601-voicemos2026-api.hf.space` · Swagger: `/docs`

**Bắt buộc:** bật **Internet ON** (Settings → Internet) để Kaggle gọi được HF Space.

| Track | Endpoint | Input | Output |
|---|---|---|---|
| 1 | `/track1` | `file_a` (+ `file_b`) | `acr_a` (+ `acr_b`, `ccr`) |
| 2 | `/track2` | `file` (+ `target_emotion`) | `qmos`, `emos`, `cat{5}`, `vad{v,a,d}` |
| 3 | `/track3` | `file_test` + `file_ref` | `spk_sim`, `acc_sim`, `cosine` |

⚠️ HF **free CPU chậm** (~vài chục giây/file; lần đầu mỗi track tải model ~1–2 phút). Notebook có **resume** nên cứ chạy lại nếu Space ngủ/timeout.

## 1. Cấu hình

In [ ]:
import os, csv, json, time, uuid, urllib.request, urllib.error

API_BASE = "https://tranminhtoan140601-voicemos2026-api.hf.space"  # đổi nếu chạy service nơi khác
EMOTIONS5 = ["angry", "happy", "neutral", "sad", "surprised"]

# Thư mục .wav cần chấm — sửa cho đúng dataset bạn Add Input
WAV_DIR  = "/kaggle/input/<your-dataset-slug>/wav"
OUT_DIR  = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
print("WAV_DIR =", WAV_DIR)

## 2. Hàm gọi API (urllib thuần — không cần `requests`)

In [ ]:
def _post_multipart(url, files, fields=None, timeout=600):
    """files: {field: path} · fields: {name: str}. Trả JSON."""
    boundary = uuid.uuid4().hex
    body = bytearray()
    for name, val in (fields or {}).items():
        if val is None:
            continue
        body += (f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"\r\n\r\n{val}\r\n').encode()
    for name, path in files.items():
        with open(path, "rb") as f:
            data = f.read()
        fn = os.path.basename(path)
        body += (f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"; filename="{fn}"'
                 f'\r\nContent-Type: audio/wav\r\n\r\n').encode()
        body += data + b"\r\n"
    body += f"--{boundary}--\r\n".encode()
    req = urllib.request.Request(url, data=bytes(body),
                                 headers={"Content-Type": f"multipart/form-data; boundary={boundary}"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def call_track1(wav_a, wav_b=None, timeout=600):
    files = {"file_a": wav_a}
    if wav_b: files["file_b"] = wav_b
    return _post_multipart(API_BASE + "/track1", files, timeout=timeout)

def call_track2(wav, target=None, timeout=600):
    return _post_multipart(API_BASE + "/track2", {"file": wav}, {"target_emotion": target}, timeout=timeout)

def call_track3(wav_test, wav_ref, timeout=600):
    return _post_multipart(API_BASE + "/track3", {"file_test": wav_test, "file_ref": wav_ref}, timeout=timeout)

# health check
try:
    with urllib.request.urlopen(API_BASE + "/health", timeout=30) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("⚠️ /health lỗi (Space có thể đang ngủ):", e)

## 3. Smoke test — gọi thử 1 file (Track 2)
Kiểm API trả đúng trước khi chạy cả thư mục. Lần đầu **chậm** vì Space tải model.

In [ ]:
wavs = sorted(f for f in os.listdir(WAV_DIR) if f.lower().endswith('.wav'))
print(f"Tìm thấy {len(wavs)} file .wav")
if wavs:
    t0 = time.time()
    res = call_track2(os.path.join(WAV_DIR, wavs[0]), target=None)
    print(f"({time.time()-t0:.1f}s)", json.dumps(res, ensure_ascii=False, indent=2))

## 4. Batch + resume cho cả thư mục
Chạy lại an toàn: file đã có trong CSV (và không lỗi) sẽ được bỏ qua.

In [ ]:
def _load_done(out_path):
    done = set()
    if os.path.exists(out_path):
        with open(out_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                if row.get("file") and not row.get("error"):
                    done.add(row["file"])
    return done

def batch_track2(wav_dir, out_csv, target=None):
    files = sorted(f for f in os.listdir(wav_dir) if f.lower().endswith('.wav'))
    cols = (["file", "qmos", "perceived_emotion"]
            + (["emos", "target", "emos_match"] if target else [])
            + [f"cat_{e}" for e in EMOTIONS5]
            + ["valence", "arousal", "dominance", "error"])
    done = _load_done(out_csv)
    print(f"{len(files)} file · target={target or '(không)'} · resume {len(done)}")
    new = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if new: w.writeheader()
        for i, name in enumerate(files, 1):
            if name in done: continue
            row = {"file": name}; t0 = time.time()
            try:
                res = call_track2(os.path.join(wav_dir, name), target, timeout=600 if i == 1 else 180)
                row["qmos"] = res.get("qmos"); row["perceived_emotion"] = res.get("perceived_emotion")
                for e in EMOTIONS5: row[f"cat_{e}"] = res.get("cat", {}).get(e)
                vad = res.get("vad", {})
                row["valence"], row["arousal"], row["dominance"] = vad.get("valence"), vad.get("arousal"), vad.get("dominance")
                if target:
                    row["emos"], row["target"], row["emos_match"] = res.get("emos"), res.get("target_emotion"), res.get("emos_match")
                print(f"[{i}/{len(files)}] {name:18s} emo={row['perceived_emotion']:9s} qmos={row['qmos']} "
                      f"V/A/D={row['valence']}/{row['arousal']}/{row['dominance']} ({time.time()-t0:.1f}s)")
            except Exception as e:
                row["error"] = repr(e)[:160]; print(f"[{i}/{len(files)}] {name} ❌ {row['error']}")
            w.writerow(row); f.flush()
    print("✅ xong →", out_csv)

# ▶️ chạy Track 2 (đổi target=... nếu muốn EMOS)
batch_track2(WAV_DIR, os.path.join(OUT_DIR, "track2_scores.csv"), target=None)

## 5. (tùy chọn) Track 1 — ACR từng file

In [ ]:
def batch_track1(wav_dir, out_csv):
    files = sorted(f for f in os.listdir(wav_dir) if f.lower().endswith('.wav'))
    cols = ["file", "acr", "error"]; done = _load_done(out_csv)
    print(f"{len(files)} file · resume {len(done)}")
    new = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if new: w.writeheader()
        for i, name in enumerate(files, 1):
            if name in done: continue
            row = {"file": name}; t0 = time.time()
            try:
                res = call_track1(os.path.join(wav_dir, name), timeout=600 if i == 1 else 180)
                row["acr"] = res.get("acr_a", res.get("acr"))
                print(f"[{i}/{len(files)}] {name:18s} acr={row['acr']} ({time.time()-t0:.1f}s)")
            except Exception as e:
                row["error"] = repr(e)[:160]; print(f"[{i}/{len(files)}] {name} ❌ {row['error']}")
            w.writerow(row); f.flush()
    print("✅ xong →", out_csv)

# batch_track1(WAV_DIR, os.path.join(OUT_DIR, "track1_scores.csv"))

## 6. (tùy chọn) Track 3 — speaker/accent similarity so với 1 reference

In [ ]:
REF_WAV = "/kaggle/input/<your-dataset-slug>/reference.wav"  # sửa đường dẫn reference

def batch_track3(wav_dir, ref_wav, out_csv):
    files = sorted(f for f in os.listdir(wav_dir) if f.lower().endswith('.wav'))
    cols = ["file", "spk_sim", "acc_sim", "cosine", "error"]; done = _load_done(out_csv)
    print(f"{len(files)} file · ref={os.path.basename(ref_wav)} · resume {len(done)}")
    new = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if new: w.writeheader()
        for i, name in enumerate(files, 1):
            if name in done: continue
            row = {"file": name}; t0 = time.time()
            try:
                res = call_track3(os.path.join(wav_dir, name), ref_wav, timeout=600 if i == 1 else 180)
                row["spk_sim"], row["acc_sim"], row["cosine"] = res.get("spk_sim"), res.get("acc_sim"), res.get("cosine")
                print(f"[{i}/{len(files)}] {name:18s} spk={row['spk_sim']} acc={row['acc_sim']} ({time.time()-t0:.1f}s)")
            except Exception as e:
                row["error"] = repr(e)[:160]; print(f"[{i}/{len(files)}] {name} ❌ {row['error']}")
            w.writerow(row); f.flush()
    print("✅ xong →", out_csv)

# batch_track3(WAV_DIR, REF_WAV, os.path.join(OUT_DIR, "track3_scores.csv"))

## 7. Xem kết quả

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(OUT_DIR, "track2_scores.csv"))
print("Tổng:", len(df), "| lỗi:", df['error'].notna().sum() if 'error' in df else 0)
print("Phân bố cảm xúc:\n", df['perceived_emotion'].value_counts())
df.head(20)